## グローバーアルゴリズムの実装（$N=2^6$の場合）
ではここから、実際にグローバーアルゴリズムを実装してデータベースの検索問題に取り掛かってみましょう。

ここで考える問題は、$N=2^6$個の要素を持つリスト（$=[0,1,2,\cdots,63]$）から、一つの答え"45"を見つけるグローバーアルゴリズムの実装です（もちろんこの数はなんでも良いので、後で自由に変更して遊んでみてください）。つまり6量子ビットの量子回路を使って、$|45\rangle=|101101\rangle$を探す問題です。

### 最初に以下の2つのセルを実行しておいてください。


In [ ]:
import sys
import shutil
import tarfile
from google.colab import drive
drive.mount('/content/gdrive')
shutil.copy('/content/gdrive/MyDrive/qcintro.tar.gz', '.')
with tarfile.open('qcintro.tar.gz', 'r:gz') as tar:
    tar.extractall(path='/root/.local')

sys.path.append('/root/.local/lib/python3.12/site-packages')

!git clone -b branch-2026 https://github.com/UTokyo-ICEPP/qc-workbook-lecturenotes
!cp -r qc-workbook-lecturenotes/qc_workbook /root/.local/lib/python3.12/site-packages/

In [ ]:
%pip install qiskit-algorithms

In [1]:
import matplotlib.pyplot as plt
import numpy as np

# Qiskit関連のパッケージをインポート
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.primitives import StatevectorSampler

from qiskit_algorithms import IterativeAmplitudeEstimation, EstimationProblem
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSampler
from qc_workbook.utils import operational_backend
from qiskit.visualization import plot_distribution

### 次のセルまで実行しておいてください。

In [ ]:
from qc_workbook.grover import make_grover_circuit

n_qubits = 6

# 量子コンピュータで実行する場合
runtime_config_path = '/content/gdrive/MyDrive/qiskit-ibm.json'
service = QiskitRuntimeService(filename=runtime_config_path)
backend = service.least_busy(filters=operational_backend())

print(f'Job will run on {backend.name}')

grover_circuit = make_grover_circuit(n_qubits)
grover_circuit_transpiled = transpile([grover_circuit], backend=backend, optimization_level=3)

sampler = RuntimeSampler(backend)
job_ibmq = sampler.run([grover_circuit_transpiled], shots=10000)
print(f">>> Job ID: {job_ibmq.job_id()}, Status: {job_ibmq.status()}")

### グローバー探索の量子回路を実装する

6量子ビットの回路`grover_circuit`を準備します。

グローバー反復を一回実行する量子回路は以下のような構成になりますが、赤枠で囲んだ部分（オラクルとDiffuserの中の$2|0\rangle\langle 0|-I$の部分）を実装する量子回路を書いてください。

一様な重ね合わせ状態$|s\rangle$を生成した後に、オラクルを実装します。

In [ ]:
Nsol = 45
n_qubits = 6

grover_circuit = QuantumCircuit(n_qubits)

grover_circuit.h(range(n_qubits))
grover_circuit.barrier()

# オラクルを作成して、回路に実装
oracle = QuantumCircuit(n_qubits)

##################
### EDIT BELOW ###
##################

#oracle.?

##################
### EDIT ABOVE ###
##################

oracle_gate = oracle.to_gate()
oracle_gate.name = "U_w"
oracle.draw('mpl')

次に、Diffuser用の回路を実装します。

In [ ]:
def diffuser(n):
    qc = QuantumCircuit(n)

    qc.h(range(n))

    ##################
    ### EDIT BELOW ###
    ##################

    #qc.?

    ##################
    ### EDIT ABOVE ###
    ##################

    qc.h(range(n))

    #print(qc)
    U_s = qc.to_gate()
    U_s.name = "U_s"
    return U_s

grover_circuit.append(oracle_gate, list(range(n_qubits)))
grover_circuit.barrier()
grover_circuit.append(diffuser(n_qubits), list(range(n_qubits)))
grover_circuit.measure_all()
grover_circuit.decompose().draw('mpl')

### シミュレータでの実験

In [ ]:
# Instantiate new AerSimulator and Sampler objects
simulator = AerSimulator()
sampler = AerSampler()

# Now run the job and examine the results
grover_circuit_transpiled = transpile(grover_circuit, backend=simulator)
sampler_sim_job = sampler.run([grover_circuit_transpiled], shots=10000)

In [ ]:
result = sampler_sim_job.result()[0]
plt.style.use('dark_background')
plot_distribution(result.data.meas.get_counts())

### 振幅増幅を確認する

では次に、グローバーのアルゴリズムを繰り返し使うことで、振幅が増幅していく様子をシミュレータを使って見てみましょう。

In [ ]:
# 繰り返しの回数
Niter = 3

grover_circuit_iterN = QuantumCircuit(n_qubits)
grover_circuit_iterN.h(range(n_qubits))
grover_circuit_iterN.barrier()
for I in range(Niter):
    grover_circuit_iterN.append(oracle_gate, list(range(n_qubits)))
    grover_circuit_iterN.barrier()
    grover_circuit_iterN.append(diffuser(n_qubits), list(range(n_qubits)))
    grover_circuit_iterN.barrier()
grover_circuit_iterN.measure_all()
grover_circuit_iterN.decompose().draw('mpl')

In [ ]:
grover_circuit_iterN_transpiled = transpile(grover_circuit_iterN, backend=simulator)
sampler_job = sampler.run([grover_circuit_iterN_transpiled], shots=10000)

result = sampler_job.result()[0]
plot_distribution(result.data.meas.get_counts())

では次に、実装した回路を繰り返し実行して、求める解を観測した回数と反復した回数との相関関係を図にしてみます。

In [ ]:
simulator = AerSimulator()
sampler = AerSampler()

x = []
y = []

shots = 10000

# 例えば10回繰り返す
for Niter in range(1,11):
    grover_circuit_iterN = QuantumCircuit(n_qubits)
    grover_circuit_iterN.h(range(n_qubits))
    for I in range(Niter):
        grover_circuit_iterN.append(oracle_gate, list(range(n_qubits)))
        grover_circuit_iterN.append(diffuser(n_qubits), list(range(n_qubits)))
    grover_circuit_iterN.measure_all()
    #print(grover_circuit_iterN)

    grover_circuit_iterN_transpiled = transpile(grover_circuit_iterN, backend=simulator)
    sampler_job_iterN = sampler.run([grover_circuit_iterN_transpiled], shots=shots)
    results_sim_iterN = sampler_job_iterN.result()[0]
    
    x.append(Niter)
    y.append(results_sim_iterN.data.meas.get_counts()[bin(Nsol)[2:]])
    
plt.clf()
plt.scatter(x,y)
plt.xlabel('N_iterations')
plt.ylabel('# of correct observations (1 solution)')
plt.show()

この図から、グローバー反復を5~6回程度繰り返すことで、正しい答えを最も高い確率で測定できることが分かりますね。計算で求めた検索に必要な反復回数と一致しているかどうか、確認してみてください。

次に、解が一つの場合で、探索リストのサイズを$N=2^4$から$N=2^{10}$まで変えた時に、測定で求めた最適な反復回数が$N$とどういう関係になっているのか調べてみましょう。

求める解は13としてみます。

In [ ]:
Nsol = 13  # =[1101]

x_Niter = []
y_Niter = []

shots = 10000

量子ビット数が4から11までの回路を作り、グローバー探索を行います。

In [ ]:
# 量子ビット数が4から11までの回路を作り、グローバー探索を行う。
for n_qubits in range(4, 11):

    # 量子ビット数を変えて回路を作る
    oracle_13 = QuantumCircuit(n_qubits)

    oracle_13.x(1)
    if n_qubits > 4:
        for i in range(4, n_qubits): oracle_13.x(i)
    oracle_13.mcp(np.pi, list(range(n_qubits - 1)), n_qubits - 1)
    oracle_13.x(1)
    if n_qubits > 4:
        for i in range(4, n_qubits): oracle_13.x(i)

    oracle_13_gate = oracle_13.to_gate()
    oracle_13_gate.name = "U_w(13)"

    # グローバー探索の結果を保存
    x = []
    y = []
    for Niter in range(1, 20):
        grover_circuit_iterN = QuantumCircuit(n_qubits)
        grover_circuit_iterN.h(range(n_qubits))
        for I in range(Niter):
            grover_circuit_iterN.append(oracle_13_gate, list(range(n_qubits)))
            grover_circuit_iterN.append(diffuser(n_qubits), list(range(n_qubits)))
        grover_circuit_iterN.measure_all()

        grover_circuit_iterN_transpiled = transpile(grover_circuit_iterN, backend=simulator)
        sampler_job_iterN = sampler.run([grover_circuit_iterN_transpiled], shots=shots)
        results_sim_iterN = sampler_job_iterN.result()[0]

        x.append(Niter)
        counts = results_sim_iterN.data.meas.get_counts()
        index = format(Nsol, f'#0{n_qubits+2}b')[2:]
        y.append(counts[index])

    plt.clf()
    plt.scatter(x,y, label=str(n_qubits)+' qubits')
    plt.xlabel('N_iterations')
    plt.ylabel('# of correct observations (1 solution)')
    plt.legend()
    plt.show()

    # 最も正しい答えを見つけるのに必要な反復回数を保存
    if n_qubits >= 4 and n_qubits <= 9: # 10以上は最大値を取るN_iterが20を超えるので、とりあえず9まで
        x_Niter.append(n_qubits)
        # 極大が複数出る場合、最初の方を選ぶ
        if n_qubits == 4:
            y_Niter.append(y.index(max(y[:5]))+1) 
        elif n_qubits == 5:
            y_Niter.append(y.index(max(y[:6]))+1) 
        elif n_qubits == 6:
            y_Niter.append(y.index(max(y[:10]))+1) 
        else:
            y_Niter.append(y.index(max(y))+1)
    
    print(f'#qubits = {n_qubits}')

探索リストのサイズと反復回数の関係を図示する

In [ ]:
array_x = np.power(2,x_Niter)
array_y = np.array(y_Niter)

# y=sqrt(x)でフィットする
from scipy.optimize import curve_fit
def sqrt_fit(x,a):
    return  a * np.sqrt(x)
param, cov = curve_fit(sqrt_fit, array_x, array_y)
value_x = np.linspace(array_x[0],array_x[len(array_x)-1],100)
value_y = param[0] * np.sqrt(value_x)

plt.clf()
plt.scatter(array_x, array_y)
plt.plot(value_x, value_y)
plt.xlabel('Size of database (= 2^n_qubits)')
plt.ylabel('# of iterations to find solution (1 solution)')
plt.show()

### 複数解の探索の場合

では次に、複数の解を探索する問題に進んでみましょう。2つの整数$x_1$と$x_2$を見つける問題へ量子回路を拡張して、求める解を観測した回数と反復した回数との相関関係を図にしてみます。

例えば、$x_1=45$と$x_2=26$の場合は

In [ ]:
n_qubits = 6

N1 = 45
N2 = 26

# 45
oracle_2sol_1 = QuantumCircuit(n_qubits)
oracle_2sol_1.x(1)
oracle_2sol_1.x(4)
oracle_2sol_1.mcp(np.pi, list(range(n_qubits-1)), n_qubits-1)
oracle_2sol_1.x(1)
oracle_2sol_1.x(4)

# 26
oracle_2sol_2 = QuantumCircuit(n_qubits)
oracle_2sol_2.x(0)
oracle_2sol_2.x(2)
oracle_2sol_2.x(5)
oracle_2sol_2.mcp(np.pi, list(range(n_qubits-1)), n_qubits-1)
oracle_2sol_2.x(0)
oracle_2sol_2.x(2)
oracle_2sol_2.x(5)

oracle_2sol_gate = QuantumCircuit(n_qubits)
oracle_2sol_gate.append(oracle_2sol_1.to_gate(), list(range(n_qubits)))
oracle_2sol_gate.barrier()
oracle_2sol_gate.append(oracle_2sol_2.to_gate(), list(range(n_qubits)))
oracle_2sol_gate.barrier()
oracle_2sol_gate.name = "U_w(2sol)"
oracle_2sol_gate.decompose().draw('mpl')

In [ ]:
x = []
y = []

for Niter in range(1,11):
    grover_circuit_2sol_iterN = QuantumCircuit(n_qubits)
    grover_circuit_2sol_iterN.h(range(n_qubits))
    for I in range(Niter):
        grover_circuit_2sol_iterN.append(oracle_2sol_gate, list(range(n_qubits)))
        grover_circuit_2sol_iterN.append(diffuser(n_qubits), list(range(n_qubits)))
    grover_circuit_2sol_iterN.measure_all()
    #print('-----  Niter =',Niter,' -----------')
    #print(grover_circuit_2sol_iterN)

    grover_circuit_2sol_iterN_transpiled = transpile(grover_circuit_2sol_iterN, backend=simulator)
    sampler_job_2sol_iterN = sampler.run([grover_circuit_2sol_iterN_transpiled], shots=shots)
    results_sim_2sol_iterN = sampler_job_2sol_iterN.result()[0]

    x.append(Niter)
    counts = results_sim_2sol_iterN.data.meas.get_counts()
    index1 = format(N1, f'#0{n_qubits+2}b')[2:]
    index2 = format(N2, f'#0{n_qubits+2}b')[2:]
    y.append(counts[index1]+counts[index2])

plt.clf()
plt.scatter(x,y)
plt.xlabel('N_iterations')
plt.ylabel('# of correct observations (2 solutions)')
plt.show()

### 量子コンピュータでの実験

グローバー反復を一回実行する回路は最初に実機で実行していたので、その結果を取ってきてシミュレーションの結果と比較します。

実機のジョブの状況を確認

In [ ]:
# Use a job id from previous results
#job_ibmq = service.job("d9078rg6c68s73ahogpg")
print(f">>> Job Status: {job_ibmq.status()}")

終わっていた場合は、シミュレーションの結果と比較

In [ ]:
print('Simulator')
result = sampler_sim_job.result()[0]
plot_distribution(result.data.meas.get_counts())

In [ ]:
print(f"IBM backend: {backend.name}, Job ID: {job_ibmq.job_id()}")
result = job_ibmq.result()
plot_distribution(result[0].data.meas.get_counts())

## 反復量子振幅推定（Iterative Quantum Amplitude Estimation (IQAE)）
ここでは、反復量子振幅推定アルゴリズムを使って振幅推定の例を見てみます。例として取り上げるのは、1) ハイゼンベルク模型での時間発展状態、2) 量子センシングを想定した微小信号による状態変化です。

### ハミルトニアン時間発展での振幅推定
横磁場イジング模型での時間発展状態の量子振幅を、反復量子振幅推定アルゴリズムを使って推定してみます。

In [ ]:
# ハミルトニアンの定義 (1D ハイゼンベルク模型)
# -------------------------------------
# スピン数
n_qubits = 6
# 結合の強さ
J = 1.5
# 境界条件
PBC = True

# ハミルトニアン生成
def heisenberg_hamiltonian(n, J=1.0, pbc=False):
    paulis = []
    coeffs = []

    if pbc:
        pairs = [(i, (i + 1) % n) for i in range(n)]
    else:
        pairs = [(i, i + 1) for i in range(n - 1)]

    for i, j in pairs:
      for p in ['X', 'Y', 'Z']:
        s = ['I']*n
        s[i] = p
        s[j] = p
        paulis.append("".join(s))
        coeffs.append(J)

    H = SparsePauliOp(paulis, coeffs)
    return H

hamiltonian = heisenberg_hamiltonian(n_qubits, J, PBC)


# 時間発展演算子と状態準備回路
# -----------------------
evolution_time = 3.0
# 1次のLie-Trotter-Suzuki分解
evolution_gate = PauliEvolutionGate(hamiltonian, time=evolution_time)

# 初期状態|000011>から時間発展させる
qc = QuantumCircuit(n_qubits)
qc.x(0)
qc.x(1)
qc.append(evolution_gate, range(n_qubits))


# 推定問題の定義
#   ターゲット:|110000>状態への遷移確率
# ---------------------------------
# objective_qubitsにリストを渡すと、指定した全ビットが '1' である状態がターゲット

# 測定基底を変える:|110000>を|111111>に変換
qc_measure = qc.copy()
qc_measure.x(range(4))

problem = EstimationProblem(
    state_preparation=qc_measure,
    objective_qubits=[0,1,2,3,4,5]
)

In [ ]:
# 反復量子振幅推定 (IQAE) の実行
# ---------------------------
sampler = StatevectorSampler()
iae = IterativeAmplitudeEstimation(
    epsilon_target = 0.001,
    alpha = 0.05,
    sampler = sampler
)
result = iae.estimate(problem)


# 結果の抽出と厳密値との比較
# ----------------------
# StatevectorSamplerを用いて厳密な状態ベクトルから真の遷移確率を計算
exact_sv = Statevector(qc)
exact_prob = np.abs(exact_sv.data[48])**2 # インデックス48が|110000>

print("--- Estimation Results ---")
print(f"Target State Probability (Exact) : {exact_prob:.5f}")
print(f"Estimated Probability (IQAE)     : {result.estimation:.5f}")
print(f"Total Oracle Queries             : {result.num_oracle_queries}")

In [ ]:
# 可視化 (振幅/確率スケールでのプロット)
# ---------------------------------
k_values = result.powers
theta_intervals = result.theta_intervals

amplitude_intervals = []
for low, high in theta_intervals:
    a_low = np.sin(2 * np.pi * low)**2
    a_high = np.sin(2 * np.pi * high)**2
    amplitude_intervals.append((a_low, a_high))

fig, ax = plt.subplots(figsize=(8, 4))
for i, (low, high) in enumerate(amplitude_intervals):
    ax.plot([low, high], [i, i], marker='o', color='tab:blue')

ax.axvline(exact_prob, color='tab:red', linestyle='--', label=f'Exact Prob: {exact_prob:.4f}')
ax.set_xlabel("Transition Probability to |110000>")
ax.set_ylabel("Iteration")
ax.set_title("IQAE: 1D TFIM Transition Probability Interval Narrowing")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(len(k_values)), k_values, marker='o', color='tab:orange')
ax.set_xlabel("Iteration")
ax.set_ylabel("k (Grover powers)")
ax.set_title("Growth of Grover Powers per Iteration")
plt.tight_layout()
plt.show()

### 微小信号の振幅推定
外部場が量子ビットに微小なラビ振動を起こすケース（例えば、量子センシング）を想定して、その確率振幅を反復量子振幅推定アルゴリズムを使って推定してみます。

In [ ]:
# 微小パラメータの設定 (量子センシングモデル)
# ------------------------------------
# 外部場との極めて弱い相互作用と時間の積を、微小な回転角θとしてエンコードする
# 例えば、結合定数が極めて小さい未知の粒子の影響
theta_physical = 0.01  # 小さな回転角

# 1量子ビットのセンシング回路
qc = QuantumCircuit(1)
# 微小信号をY軸回転として状態に変換
qc.ry(theta_physical, 0)

# 厳密な振幅と確率
exact_amplitude = np.sin(theta_physical/2)
exact_prob = exact_amplitude**2

# 推定問題の定義
# ------------
# 目標状態は|1>
# この振幅a = sin(θ/2)を高精度で推定する
problem = EstimationProblem(
    state_preparation = qc,
    objective_qubits = [0]
)


# 高精度推定を目指したIQAEの実行
# --------------------------
# 目標精度を1e-4に設定（古典サンプリングとの比較を行う）
target_precision = 1e-4

sampler = StatevectorSampler()
iae = IterativeAmplitudeEstimation(
    epsilon_target = target_precision,
    alpha = 0.05,  # 95% 信頼区間
    sampler = sampler
)

print(f"Target Amplitude to estimate: {exact_amplitude:.4e}")
print(f"Target Precision (epsilon):   {target_precision:.4e}\n")

In [ ]:
print("Running IQAE... (This could take some time due to deep Grover iterations)\n")
result = iae.estimate(problem)


# 結果の抽出とリソース評価
# --------------------
# 古典的なモンテカルロ法(ショットサンプリング)で同精度を出すのに必要な概算ショット数
# (Chernoff bound: N ≈ 1 / (epsilon^2))
classical_shots_needed = int(1 / (target_precision**2))

print("--- Estimation Results ---")
print(f"Target Probability (Exact)    : {exact_prob:.8e}")
print(f"Estimated Probability (IQAE)  : {result.estimation:.8e}")
print(f"Absoute Estimated Error       : {abs(exact_prob - result.estimation):.8e}\n")

print("--- Resource Comparison ---")
print(f"Quantum Oracle Queries (IQAE)       : {result.num_oracle_queries:,}")
print(f"Equivalent Classical Shots          : ~{classical_shots_needed:,}")
if result.num_oracle_queries > 0:
    ratio = classical_shots_needed / result.num_oracle_queries
    print(f"Classical Shot/Quantum Query Ratio  : ~{ratio:,.1f}")

In [ ]:
# 可視化 (微小スケールでの区間収束)
# ----------------------------
k_values = result.powers
theta_intervals = result.theta_intervals

# "a = sin^2(2 * pi * theta)"に基づいて振幅に変換
amplitude_intervals = []
for low, high in theta_intervals:
    a_low = np.sin(2 * np.pi * low)**2
    a_high = np.sin(2 * np.pi * high)**2
    amplitude_intervals.append((a_low, a_high))

iterations = range(len(k_values))

fig, ax1 = plt.subplots(figsize=(9, 5))

# 対数スケールで見やすいように、真の値からの絶対誤差（信頼区間の幅）をプロット
interval_widths = [high - low for low, high in amplitude_intervals]
ax1.plot(iterations, interval_widths, marker='o', color='tab:blue', label="Confidence Interval Width")
ax1.axhline(exact_amplitude*target_precision, color='tab:red', linestyle='--', label="Target Precision ($\\epsilon$)")

ax1.set_yscale('log')
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Amplitude Interval Width (Log Scale)")
ax1.set_title("IQAE: Convergence of Precision for Minute Parameter")
ax1.legend()
plt.tight_layout()
plt.show()

# Grover演算回数の成長を対数スケールでプロット
fig, ax2 = plt.subplots(figsize=(9, 5))
ax2.plot(iterations, k_values, marker='o', color='tab:orange')
ax2.set_yscale('log')
ax2.set_xlabel("Iteration")
ax2.set_ylabel("k (Grover powers) - Log Scale")
ax2.set_title("Exponential Growth of Oracle Queries to Reach High Precision")
plt.tight_layout()
plt.show()

**上の問題の回答**

オラクルの中身

In [ ]:
oracle.x(1)
oracle.x(4)
oracle.mcp(np.pi, list(range(n_qubits-1)), n_qubits-1)
oracle.x(1)
oracle.x(4)

Diffuserの中身

In [ ]:
    qc.rz(2*np.pi, n-1)
    qc.x(list(range(n)))

    # multi-controlled Zゲート
    qc.mcp(np.pi, list(range(n-1)), n-1)

    qc.x(list(range(n)))